# 00 · Scenarios: load and look

Starter notebook for Phase 4. It shows how to get a generator scenario into a notebook and plot it.
It contains **no analysis**: the pipeline steps (`notebooks/01_detrend.ipynb` onwards) are yours.

**Blind rule.** Your pipeline code only ever gets `scenario.light_curve` (time, flux, flux_err).
The answer key, the noise-free model and spot-free twins are for learning and checking, and live
in the clearly marked section at the bottom.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from exoplanet_lab.generator import Preset, generate_preset_scenario

PRESET = Preset.MEDIUM
SEED = 3

scenario = generate_preset_scenario(PRESET, SEED)
light_curve = scenario.light_curve  # the only thing the pipeline sees
n_points, span = light_curve.time.size, light_curve.time[-1] - light_curve.time[0]
print(f"{n_points} points over {span:.2f} d, median flux {np.median(light_curve.flux):.6f}")

## Raw light curve

In [ ]:
fig, (full, zoom) = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)
full.plot(light_curve.time, light_curve.flux, ".", ms=1, color="0.3")
full.set(xlabel="time [d]", ylabel="flux [norm]", title=f"{PRESET.value} · seed {SEED}")

window = (light_curve.time > 2.0) & (light_curve.time < 4.0)
zoom.plot(light_curve.time[window], light_curve.flux[window], ".", ms=2, color="0.3")
zoom.set(xlabel="time [d]", ylabel="flux [norm]", title="days 2-4")
plt.show()

---
## Learning only: peek at the truth

Never use anything below inside pipeline code. It is here so you can see what the generator
planted and, for detrending, compare against the **spot-free twin**: the same scenario with
starspots switched off. Because every random part of the generator has its own stream
(decision 15), the twin has identical noise, gaps and transits.

In [ ]:
from exoplanet_lab.generator import generate_scenario

key = scenario.answer_key
print(key.type.value, "| expected SNR:", key.expected_snr)
if key.planet is not None:
    p = key.planet
    print(
        f"P = {p.period_days:.4f} d, t0 = {p.t0_days:.4f} d, T14 = {p.duration_days * 24:.2f} h, "
        f"depth = {p.depth * 1e6:.0f} ppm, transits observed = {p.n_transits_observed}"
    )
print(
    f"spots: {key.star.activity_amplitude * 100:.2f} % peak-to-peak, "
    f"rotation {key.star.rotation_period_days:.2f} d"
)

star = key.settings.star.model_copy(update={"activity_amplitude": 0.0})
quiet = generate_scenario(key.settings.model_copy(update={"star": star}))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
ax.plot(scenario.time, scenario.flux, ".", ms=1, color="0.6", label="as observed")
ax.plot(quiet.time, quiet.flux, ".", ms=1, color="tab:blue", label="spot-free twin")
if key.planet is not None:
    p = key.planet
    for mid in np.arange(p.t0_days, scenario.time[-1], p.period_days):
        ax.axvline(mid, color="tab:orange", lw=0.6, ls="--")
ax.set(xlabel="time [d]", ylabel="flux [norm]", title="truth: twin and transit times (dashed)")
ax.legend(markerscale=8)
plt.show()